# Figure 4D (revision). Genomic traits separating Liang enriched vs unenriched viromes

**Goal.** Among ≥5× Caudoviricetes detections in the Liang paired cohort, identify genomic / quality thresholds that maximize separation between **enriched** and **unenriched** libraries, and visualize continuous distributions for the strongest separators.

**Sections**
1. Document the existing UHVDB r6 sylph → CoverM → gene-coverage profiling workflow (do not re-run here)
2. Load ≥5× Caudoviricetes detections and scan thresholds for every genomic comparator
3. Histograms of continuous values for every comparator with |gap| ≥ 10 pp
4. Violin plots for the same |gap| ≥ 10 pp separators
5. Clean selected-panel violins (fixed metric set / labels; no titles or per-panel stats)
6. Same selected-panel violins for **bulk vs HEV** and **no MMC vs MMC**


## 1. UHVDB r6 profiling of Liang paired metagenomes

Profiling is already implemented as a Slurm array job. **Do not re-submit from this notebook** unless regenerating results from scratch.

| Item | Path |
|---|---|
| Submit script | `activity_profiling/liang_enriched_r6_sylph_coverm.sbatch` |
| Sample sheet | `activity_profiling/liang_enriched_paired_samplesheet.csv` |
| SPRING inputs | `activity_profiling/liang_enriched_results/2026-04-03_outputs/preprocess/<sample>.spring` |
| Results root | `activity_profiling/liang_enriched_r6/results/<sample>/` |
| Logs | `activity_profiling/liang_enriched_r6/logs/` |
| ≥5× genomic table (used below) | `activity_profiling/liang_enriched_r6/genomic_characteristics_ge5x/detections_ge5x.tsv` |

**Workflow (per sample)**
1. Decompress SPRING → FASTQ
2. `sylph profile` against **UHVDB r6** species-rep sketch (`-c 200`) + **GTDB r226**
3. Select contained UHVDB genomes from the profile
4. `coverm contig` (strobealign) → trimmed_mean / mean / breadth
5. `uhvdb_genecoverage.py` on the CoverM BAM

**Submit (from `activity_profiling/`)**
```bash
sbatch liang_enriched_r6_sylph_coverm.sbatch
```
Array `0–75%25` on `ckpt`; failed tasks auto-retry once on `compute-hugemem`.

Downstream ≥5× Caudoviricetes feature tables were built into `genomic_characteristics_ge5x/` (same metric panel as the bulk/MMC genomic comparisons).


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from scipy import stats as scipy_stats

plt.rcParams.update({"font.size": 12})

ROOT = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB")
FIG4 = ROOT / "uhvdb-manuscript/figure_4"
LIANG = FIG4 / "activity_profiling/liang_enriched_r6"
SBATCH = FIG4 / "activity_profiling/liang_enriched_r6_sylph_coverm.sbatch"
DETECTIONS = LIANG / "genomic_characteristics_ge5x/detections_ge5x.tsv"
METADATA = ROOT / "uhvdb-manuscript/figure_s18/uhvdb_v5_final_metadata_v2.tsv.gz"
OUT_DIR = FIG4 / "plots" / "figure_4d_revision"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert SBATCH.is_file(), SBATCH
assert DETECTIONS.is_file(), DETECTIONS
print("sbatch :", SBATCH)
print("results:", LIANG / "results")
print("ge5x   :", DETECTIONS)
print("out    :", OUT_DIR)


## 2. Optimal thresholds by maximum enriched − unenriched separation

For each continuous comparator in the full genomic panel:
- evaluate **≥ t** and **≤ t** over a dense grid of candidate thresholds
- **optimal** = threshold/direction maximizing `|pct_enriched − pct_unenriched|` (pure max gap; no recovery floor)
- also report Cliff’s δ on the continuous distributions (positive δ = enriched > unenriched)


In [ ]:
# Load ≥5× Caudoviricetes detections; attach CheckV AF / AAI if needed
genomic = pl.read_csv(DETECTIONS, separator="\t")
print(
    genomic.group_by("library")
    .agg(
        [
            pl.len().alias("n_detections"),
            pl.col("sample_id").n_unique().alias("n_samples"),
        ]
    )
    .sort("library")
)

need_aai = any(c not in genomic.columns for c in ("aai_af", "aai_id", "aai_id_af"))
if need_aai:
    aai_cache = OUT_DIR / "aai_lookup_for_ge5x.tsv"
    if aai_cache.is_file():
        aai = pl.read_csv(aai_cache, separator="\t")
    else:
        ids = genomic.get_column("contig_id").unique().to_list()
        aai = (
            pl.scan_csv(METADATA, separator="\t", infer_schema_length=1000)
            .select(["uhvdb_id", "seq_name", "seqhash_rep", "aai_id", "aai_af"])
            .filter(
                (pl.col("seq_name") == pl.col("seqhash_rep"))
                & pl.col("uhvdb_id").is_in(ids)
            )
            .select(
                [
                    "uhvdb_id",
                    pl.col("aai_id").cast(pl.Float64),
                    pl.col("aai_af").cast(pl.Float64),
                ]
            )
            .unique(subset=["uhvdb_id"], keep="first")
            .collect()
            .with_columns(((pl.col("aai_id") / 100.0) * pl.col("aai_af")).alias("aai_id_af"))
        )
        aai.write_csv(aai_cache, separator="\t")
        print("wrote", aai_cache)
    drop_cols = [c for c in ("aai_id", "aai_af", "aai_id_af") if c in genomic.columns]
    if drop_cols:
        genomic = genomic.drop(drop_cols)
    genomic = genomic.join(aai, left_on="contig_id", right_on="uhvdb_id", how="left")

une = genomic.filter(pl.col("library") == "unenriched")
enr = genomic.filter(pl.col("library") == "enriched")
print(f"unenriched: {une.height:,}  enriched: {enr.height:,}")


In [ ]:
# Full genomic panel (continuous comparators)
COMPARATORS = [
    ("breadth", "Breadth"),
    ("mean", "Mean coverage"),
    ("virus_score", "geNomad virus score"),
    ("virulent", "Virulent score"),
    ("complete_count", "Complete genomes in UHVDB species"),
    ("prop_integrated_in_species", "Proportion integrated in species"),
    ("n_conserved_hallmarks", "Conserved hallmarks (MCP+TerL+portal)"),
    ("n_hallmarks", "geNomad hallmark count"),
    ("n_hallmarks_per_10kb", "geNomad hallmarks / 10 kb"),
    ("virus_hallmarks", "geNomad virus hallmark genes"),
    ("num_capsid", "Capsid gene count"),
    ("num_tail", "Tail gene count"),
    ("num_lysis", "Lysis gene count"),
    ("capsid_per_10kb", "Capsid genes / 10 kb"),
    ("tail_per_10kb", "Tail genes / 10 kb"),
    ("lysis_per_10kb", "Lysis genes / 10 kb"),
    ("host_genes", "CheckV host gene count"),
    ("host_genes_per_10kb", "CheckV host genes / 10 kb"),
    ("viral_genes", "CheckV viral gene count"),
    ("viral_genes_per_10kb", "CheckV viral genes / 10 kb"),
    ("n_integration", "Integration gene count"),
    ("integration_per_10kb", "Integration genes / 10 kb"),
    ("n_amg_host_takeover", "AMG / host-takeover gene count"),
    ("amg_host_takeover_per_10kb", "AMG / host-takeover genes / 10 kb"),
    ("n_dna_metabolism", "DNA metabolism gene count"),
    ("dna_metabolism_per_10kb", "DNA metabolism genes / 10 kb"),
    ("aai_af", "CheckV AF to nearest DTR"),
    ("aai_id", "CheckV AAI to nearest DTR"),
    ("aai_id_af", "CheckV AAI × AF to nearest DTR"),
    ("genome_length", "Genome length (bp)"),
]

# Keep only columns present
COMPARATORS = [(c, lab) for c, lab in COMPARATORS if c in genomic.columns]
print(f"{len(COMPARATORS)} comparators")


In [ ]:
def _vals(df: pl.DataFrame, col: str) -> np.ndarray:
    return df.select(pl.col(col).cast(pl.Float64)).drop_nulls().to_series().to_numpy()


def cliffs_delta(a: np.ndarray, b: np.ndarray) -> float:
    """Positive δ ⇒ values in b tend to be larger than in a."""
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    u, _ = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")
    return -((2.0 * u) / (len(a) * len(b)) - 1.0)


def candidate_thresholds(values: np.ndarray, col: str) -> np.ndarray:
    """Dense threshold grid from observed values (+ common cutpoints for scores)."""
    v = values[np.isfinite(values)]
    if v.size == 0:
        return np.array([])
    uniq = np.unique(v)
    # integer-like counts: use observed ints
    if np.allclose(uniq, np.round(uniq)) and uniq.size <= 80:
        thr = uniq.copy()
    else:
        # percentiles + linspace between min/max
        qs = np.unique(np.quantile(v, np.linspace(0, 1, 101)))
        lin = np.linspace(float(v.min()), float(v.max()), 51)
        thr = np.unique(np.concatenate([qs, lin]))
    extras = {
        "virus_score": [0.9, 0.95, 0.99, 0.995, 0.999, 0.9995],
        "breadth": [0.25, 0.5, 0.75, 0.9],
        "aai_af": [20, 30, 40, 40.5, 50, 60, 70, 80, 90],
        "aai_id": [50, 70, 80, 90, 95, 99],
        "aai_id_af": [20, 25, 30, 40, 50, 60, 70, 80],
        "host_genes_per_10kb": [0, 0.5, 1, 2],
        "amg_host_takeover_per_10kb": [0, 0.5, 1],
    }
    if col in extras:
        thr = np.unique(np.concatenate([thr, np.asarray(extras[col], dtype=float)]))
    return thr


def max_gap_threshold(a: np.ndarray, b: np.ndarray, col: str) -> dict:
    """Find ≥t or ≤t maximizing |pct_b − pct_a|."""
    thr = candidate_thresholds(np.concatenate([a, b]), col)
    best = {
        "direction": None,
        "threshold": np.nan,
        "pct_unenriched": np.nan,
        "pct_enriched": np.nan,
        "gap_enriched_minus_unenriched": np.nan,
        "abs_gap": -1.0,
    }
    n_a, n_b = len(a), len(b)
    if n_a == 0 or n_b == 0 or thr.size == 0:
        return best
    for t in thr:
        for direction, pred_a, pred_b in (
            ("ge", a >= t, b >= t),
            ("le", a <= t, b <= t),
        ):
            pa = 100.0 * float(np.mean(pred_a))
            pb = 100.0 * float(np.mean(pred_b))
            gap = pb - pa
            if abs(gap) > best["abs_gap"]:
                best = {
                    "direction": direction,
                    "threshold": float(t),
                    "pct_unenriched": pa,
                    "pct_enriched": pb,
                    "gap_enriched_minus_unenriched": gap,
                    "abs_gap": abs(gap),
                }
    return best


rows = []
for col, label in COMPARATORS:
    a = _vals(une, col)
    b = _vals(enr, col)
    opt = max_gap_threshold(a, b, col)
    d = cliffs_delta(a, b)
    if len(a) and len(b):
        u, p = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")
    else:
        u, p = np.nan, np.nan
    rule = (
        f"{'≥' if opt['direction'] == 'ge' else '≤'} {opt['threshold']:g}"
        if opt["direction"]
        else ""
    )
    rows.append(
        {
            "metric": col,
            "label": label,
            "n_unenriched": int(len(a)),
            "n_enriched": int(len(b)),
            "median_unenriched": float(np.median(a)) if len(a) else np.nan,
            "median_enriched": float(np.median(b)) if len(b) else np.nan,
            "mean_unenriched": float(np.mean(a)) if len(a) else np.nan,
            "mean_enriched": float(np.mean(b)) if len(b) else np.nan,
            "cliffs_delta_enriched_minus_unenriched": d,
            "mannwhitney_p": float(p) if np.isfinite(p) else np.nan,
            "optimal_direction": opt["direction"],
            "optimal_threshold": opt["threshold"],
            "optimal_rule": rule,
            "pct_unenriched_at_opt": opt["pct_unenriched"],
            "pct_enriched_at_opt": opt["pct_enriched"],
            "gap_pp_at_opt": opt["gap_enriched_minus_unenriched"],
            "abs_gap_pp_at_opt": opt["abs_gap"],
        }
    )

thresh_df = pd.DataFrame(rows).sort_values("abs_gap_pp_at_opt", ascending=False)
thresh_path = OUT_DIR / "optimal_thresholds_max_gap.tsv"
thresh_df.to_csv(thresh_path, sep="\t", index=False)
print("wrote", thresh_path)

show_cols = [
    "label",
    "optimal_rule",
    "pct_unenriched_at_opt",
    "pct_enriched_at_opt",
    "gap_pp_at_opt",
    "abs_gap_pp_at_opt",
    "cliffs_delta_enriched_minus_unenriched",
    "median_unenriched",
    "median_enriched",
]
display(thresh_df[show_cols].round(3))


## 3. Continuous histograms for separators with |gap| ≥ 10 pp

Plot overlaid enriched vs unenriched histograms for every comparator whose optimal binary threshold achieves `|pct_enriched − pct_unenriched| ≥ 10` percentage points (sorted by `|gap|`, ties broken by `|Cliff’s δ|`).


In [ ]:
MIN_ABS_GAP_PP = 10.0  # include all separators with |gap| ≥ this
pal = {"unenriched": "#4C72B0", "enriched": "#DD8452"}

top = thresh_df.loc[thresh_df["abs_gap_pp_at_opt"] >= MIN_ABS_GAP_PP].sort_values(
    ["abs_gap_pp_at_opt", "cliffs_delta_enriched_minus_unenriched"],
    ascending=[False, False],
)
if top.empty:
    raise ValueError(f"No separators with |gap| ≥ {MIN_ABS_GAP_PP} pp")

n = len(top)
ncol = 3
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.5 * nrow), squeeze=False)

for ax, row in zip(axes.ravel(), top.itertuples(index=False)):
    a = _vals(une, row.metric)
    b = _vals(enr, row.metric)
    # shared bins
    lo = float(np.nanmin(np.concatenate([a, b])))
    hi = float(np.nanmax(np.concatenate([a, b])))
    if hi <= lo:
        hi = lo + 1.0
    # Always use shared explicit edges so unenriched and enriched match.
    # Integer counts with modest span → integer bins; else fixed shared linspace
    # (avoid exploding bins for high-range ints like mean coverage / genome length).
    uniq = np.unique(np.concatenate([a, b]))
    if uniq.size and np.allclose(uniq, np.round(uniq)) and (hi - lo) <= 80:
        bins = np.arange(np.floor(lo), np.ceil(hi) + 2) - 0.5
    else:
        bins = np.linspace(lo, hi, 31)
    ax.hist(
        a,
        bins=bins,
        density=True,
        alpha=0.55,
        color=pal["unenriched"],
        edgecolor="black",
        linewidth=0.4,
        label="Unenriched",
    )
    ax.hist(
        b,
        bins=bins,
        density=True,
        alpha=0.55,
        color=pal["enriched"],
        edgecolor="black",
        linewidth=0.4,
        label="Enriched",
    )
    # mark optimal threshold
    if np.isfinite(row.optimal_threshold):
        ax.axvline(
            row.optimal_threshold,
            color="0.15",
            linestyle="--",
            linewidth=1.2,
            label=f"opt {row.optimal_rule}",
        )
    ax.set_xlabel(row.label)
    ax.set_ylabel("Density")
    ax.set_title(
        f"|Δ|={row.abs_gap_pp_at_opt:.1f} pp  δ={row.cliffs_delta_enriched_minus_unenriched:+.2f}\n"
        f"{row.optimal_rule}: une {row.pct_unenriched_at_opt:.1f}% / enr {row.pct_enriched_at_opt:.1f}%",
        fontsize=10,
    )
    ax.grid(axis="y", alpha=0.3)

# single legend on first axis
handles, labels = axes[0, 0].get_legend_handles_labels()
# dedupe labels
seen = set()
H, L = [], []
for h, lab in zip(handles, labels):
    if lab.startswith("opt "):
        continue
    if lab not in seen:
        seen.add(lab)
        H.append(h)
        L.append(lab)
fig.legend(H, L, loc="upper center", ncol=2, frameon=True, bbox_to_anchor=(0.5, 1.02))

for ax in axes.ravel()[n:]:
    ax.axis("off")

fig.suptitle(
    "Liang paired: continuous trait distributions (≥5× Caudoviricetes)\n"
    f"Separators with |gap| ≥ {MIN_ABS_GAP_PP:.0f} pp at optimal threshold (n={n})",
    y=1.06,
    fontsize=13,
)
fig.tight_layout()
png = OUT_DIR / "figure_4d_top_separator_histograms.png"
pdf = OUT_DIR / "figure_4d_top_separator_histograms.pdf"
fig.savefig(png, dpi=200, bbox_inches="tight")
try:
    fig.savefig(pdf, bbox_inches="tight")
    print("wrote", pdf)
except Exception as exc:
    print(f"PDF skipped ({exc.__class__.__name__}: {exc})")
print("wrote", png)
plt.show()


In [ ]:
# Compact summary of separators with |gap| >= MIN_ABS_GAP_PP
summary = top[
    [
        "label",
        "optimal_rule",
        "pct_unenriched_at_opt",
        "pct_enriched_at_opt",
        "gap_pp_at_opt",
        "cliffs_delta_enriched_minus_unenriched",
    ]
].copy()
summary_path = OUT_DIR / "figure_4d_top_separators_summary.tsv"
summary.to_csv(summary_path, sep="\t", index=False)
print("wrote", summary_path)
display(summary.round(3))


## 4. Violin plots for separators with |gap| ≥ 10 pp

Same comparator set and sort order as the histograms above.


In [ ]:
# Violin plots for the same |gap| ≥ MIN_ABS_GAP_PP separators
n = len(top)
ncol = 3
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.6 * nrow), squeeze=False)

for ax, row in zip(axes.ravel(), top.itertuples(index=False)):
    a = _vals(une, row.metric)
    b = _vals(enr, row.metric)
    data = [a, b]
    parts = ax.violinplot(
        data,
        positions=[0, 1],
        showmeans=False,
        showmedians=False,
        showextrema=False,
        widths=0.9,
    )
    for body, key in zip(parts["bodies"], ["unenriched", "enriched"]):
        body.set_facecolor(pal[key])
        body.set_edgecolor("black")
        body.set_alpha(0.55)
        body.set_linewidth(0.8)
    # Overlay boxplots (median / IQR) and means
    bp = ax.boxplot(
        data,
        positions=[0, 1],
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "0.1", "linewidth": 1.3},
        whiskerprops={"color": "0.1", "linewidth": 1.0},
        capprops={"color": "0.1", "linewidth": 1.0},
        boxprops={"facecolor": "white", "edgecolor": "0.1", "linewidth": 1.0, "alpha": 0.9},
    )
    means = [float(np.mean(x)) for x in data]
    ax.scatter(
        [0, 1],
        means,
        marker="D",
        s=28,
        color="0.1",
        zorder=5,
        label="mean",
    )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Unenriched", "Enriched"])
    ax.set_ylabel(row.label)
    ax.set_title(
        f"|Δ|={row.abs_gap_pp_at_opt:.1f} pp  δ={row.cliffs_delta_enriched_minus_unenriched:+.2f}\n"
        f"{row.optimal_rule}: une {row.pct_unenriched_at_opt:.1f}% / enr {row.pct_enriched_at_opt:.1f}%",
        fontsize=10,
    )
    ax.grid(axis="y", alpha=0.3)

for ax in axes.ravel()[n:]:
    ax.axis("off")

fig.suptitle(
    "Liang paired: continuous trait violins (≥5× Caudoviricetes)\n"
    f"Separators with |gap| ≥ {MIN_ABS_GAP_PP:.0f} pp at optimal threshold (n={n})",
    y=1.04,
    fontsize=13,
)
fig.tight_layout()
png = OUT_DIR / "figure_4d_top_separator_violins.png"
pdf = OUT_DIR / "figure_4d_top_separator_violins.pdf"
fig.savefig(png, dpi=200, bbox_inches="tight")
try:
    fig.savefig(pdf, bbox_inches="tight")
    print("wrote", pdf)
except Exception as exc:
    print(f"PDF skipped ({exc.__class__.__name__}: {exc})")
print("wrote", png)
plt.show()


## 5. Selected violins (publication-style)

Paired contrast (not library-level):
- **Bulk only**: pair×virus ≥5× in unenriched and not ≥5× in paired enriched
- **Bulk + enriched**: pair×virus ≥5× in both; enriched-only excluded
- Metrics from the unenriched sample; one row per pair × virus

Fixed metric order and display labels. Nested boxplots + mean diamonds; no figure title. Significance: Mann–Whitney U (two-sided) with Benjamini–Hochberg FDR across these 11 comparisons; asterisks above each panel (`ns`, `*`, `**`, `***` for q ≥ 0.05, < 0.05, < 0.01, < 0.001).


In [ ]:
SELECTED_VIOLINS = [
    ("breadth", "Breadth"),
    ("aai_id_af", "CheckV AAI × AF to nearest DTR"),
    ("host_genes_per_10kb", "CheckV Host genes / 10 Kb"),
    ("viral_genes_per_10kb", "CheckV viral genes / 10 Kb"),
    ("lysis_per_10kb", "Lysis genes / 10 Kb"),
    ("capsid_per_10kb", "Capsid genes / 10 Kb"),
    ("tail_per_10kb", "Tail genes / 10 Kb"),
    ("amg_host_takeover_per_10kb", "AMG genes / 10 Kb"),
    ("dna_metabolism_per_10kb", "DNA metabolism genes / 10 Kb"),
    ("integration_per_10kb", "Integration genes / 10 Kb"),
    ("n_conserved_hallmarks", "MCP + TerL + portal"),
]

PAIRED_TSV = OUT_DIR / "liang_paired_unenriched_only_vs_shared_ge5x.tsv"
paired = pd.read_csv(PAIRED_TSV, sep="\t")
bulk_only = paired[paired["arm"] == "unenriched_only"]
bulk_plus = paired[paired["arm"] == "shared"]
print(f"bulk only: {len(bulk_only):,}  bulk + enriched: {len(bulk_plus):,}")
pal_paired = {"bulk_only": "#4C72B0", "bulk_plus": "#DD8452"}


def bh_qvalues(pvals: np.ndarray) -> np.ndarray:
    """Benjamini–Hochberg FDR q-values."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranked = p[order]
    q_ranked = np.empty(n)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = min(prev, ranked[i] * n / rank)
        q_ranked[i] = val
        prev = val
    out = np.empty(n)
    out[order] = q_ranked
    return np.clip(out, 0, 1)


def sig_label(q: float) -> str:
    if not np.isfinite(q):
        return "n/a"
    if q < 0.001:
        return "***"
    if q < 0.01:
        return "**"
    if q < 0.05:
        return "*"
    return "ns"


def _vals_pd(frame: pd.DataFrame, col: str) -> np.ndarray:
    v = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype=float)
    return v[np.isfinite(v)]


missing = [c for c, _ in SELECTED_VIOLINS if c not in paired.columns]
if missing:
    raise ValueError(f"Missing columns for selected violins: {missing}")

# Mann–Whitney + BH across the plotted panel
rows = []
for metric, label in SELECTED_VIOLINS:
    a = _vals_pd(bulk_only, metric)
    b = _vals_pd(bulk_plus, metric)
    _, p = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append(
        {
            "metric": metric,
            "label": label,
            "n_bulk_only": int(a.size),
            "n_bulk_plus_enriched": int(b.size),
            "mannwhitney_p": float(p),
        }
    )
sel_stats = pd.DataFrame(rows)
sel_stats["bh_q"] = bh_qvalues(sel_stats["mannwhitney_p"].to_numpy())
sel_stats["sig"] = sel_stats["bh_q"].map(sig_label)
stats_path = OUT_DIR / "figure_4d_selected_violins_stats.tsv"
sel_stats.to_csv(stats_path, sep="\t", index=False)
print("wrote", stats_path)
display(sel_stats.round(6))

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 12,
})
SIG_FS = 15

n = len(SELECTED_VIOLINS)
ncol = 3
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.8 * nrow), squeeze=False)
q_by_metric = dict(zip(sel_stats["metric"], sel_stats["bh_q"]))

for ax, (metric, label) in zip(axes.ravel(), SELECTED_VIOLINS):
    a = _vals_pd(bulk_only, metric)
    b = _vals_pd(bulk_plus, metric)
    data = [a, b]
    parts = ax.violinplot(
        data,
        positions=[0, 1],
        showmeans=False,
        showmedians=False,
        showextrema=False,
        widths=0.9,
    )
    for body, key in zip(parts["bodies"], ["bulk_only", "bulk_plus"]):
        body.set_facecolor(pal_paired[key])
        body.set_edgecolor("black")
        body.set_alpha(0.55)
        body.set_linewidth(0.8)
    ax.boxplot(
        data,
        positions=[0, 1],
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "0.1", "linewidth": 1.3},
        whiskerprops={"color": "0.1", "linewidth": 1.0},
        capprops={"color": "0.1", "linewidth": 1.0},
        boxprops={"facecolor": "white", "edgecolor": "0.1", "linewidth": 1.0, "alpha": 0.9},
    )
    means = [float(np.mean(x)) for x in data]
    ax.scatter([0, 1], means, marker="D", s=28, color="0.1", zorder=5)
    # Significance bracket (BH q across these 11 tests)
    y0, y1 = ax.get_ylim()
    span = y1 - y0 if y1 > y0 else 1.0
    y_bar = y1 + 0.04 * span
    ax.plot([0, 0, 1, 1], [y_bar - 0.015 * span, y_bar, y_bar, y_bar - 0.015 * span], color="0.1", lw=1.1)
    ax.text(0.5, y_bar + 0.01 * span, sig_label(q_by_metric[metric]), ha="center", va="bottom", fontsize=SIG_FS)
    ax.set_ylim(y0, y1 + 0.16 * span)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Bulk only", "Bulk + enriched"])
    ax.set_ylabel(label)
    ax.tick_params(axis="both", labelsize=12)
    ax.yaxis.label.set_size(14)
    ax.grid(axis="y", alpha=0.3)

for ax in axes.ravel()[n:]:
    ax.axis("off")

fig.tight_layout()
png = OUT_DIR / "figure_4d_selected_violins.png"
pdf = OUT_DIR / "figure_4d_selected_violins.pdf"
fig.savefig(png, dpi=200, bbox_inches="tight")
try:
    fig.savefig(pdf, bbox_inches="tight")
    print("wrote", pdf)
except Exception as exc:
    print(f"PDF skipped ({exc.__class__.__name__}: {exc})")
(OUT_DIR / "figure_4d_selected_violins_NOTE.txt").write_text(
    "Paired contrast for Liang selected violins:\n"
    "- Bulk only: pair×virus with mean≥5× in unenriched and NOT ≥5× in paired enriched.\n"
    "- Bulk + enriched: pair×virus with mean≥5× in BOTH unenriched and paired enriched.\n"
    "- Enriched-only viruses excluded.\n"
    "- All plotted metrics taken from the unenriched sample.\n"
    "- Unit: one row per pair × virus.\n"
    f"- n: bulk only={len(bulk_only)}, bulk + enriched={len(bulk_plus)}.\n"
)
print("wrote", png)
plt.show()


## 6. Companion selected violins (bulk vs HEV; uninduced vs induced)

Same 11-metric selected panel, nested boxplots + means, and Mann–Whitney + BH asterisks as section 5. MMC panel axis labels: **Uninduced** / **Induced** (paired induction contrast; same figure width as section 5).

| Comparison | Inputs | Outputs |
|---|---|---|
| Bulk vs HEV | `paired_virome_dataset2/.../genomic_characteristics_ge5x_with_hev/detections_ge5x.tsv` (paired bulk vs `cohort=hev`) | `figure_4d_selected_violins_bulk_vs_hev.{png,pdf,_stats.tsv}` |
| Uninduced vs Induced | `mmc_activity_analysis/genomic_characteristics_ge5x_induced/detections_ge5x.tsv` (paired; metrics from no_mmc) | `figure_4d_selected_violins_no_mmc_vs_mmc.{png,pdf,_stats.tsv}` |

**Note.** Unlike Liang r6 (gene_coverage breadth ≥ 0.8), these two companion panels currently use the existing ge5x gene-category `/10 kb` columns (annotation-based): bulk `*.gene_coverage.tsv.gz` work files are broken symlinks, and MMC gene_coverage is only available for a small mvirs/transduct subset.


## Notes

- Selected violins (section 5): paired **bulk only** vs **bulk + enriched** (≥5× unenriched-only vs ≥5× in both; enriched-only excluded; unenriched metrics; pair×virus).
- Threshold / histogram / top-separator panels still use library-level ≥5× Caudoviricetes detections from `genomic_characteristics_ge5x`.
- Gene-category counts (capsid/tail/lysis/integration/AMG/DNA metabolism and MCP+TerL+portal) are from sample `*.gene_coverage.tsv.gz` genes with **breadth ≥ 0.8**, not static genome annotations. Annotation-based table kept as `detections_ge5x.annotation_based.tsv`.
- Threshold optimality uses **pure max |gap|** between enriched and unenriched pass rates (no enriched-recovery floor).
- Both ≥ and ≤ directions are considered for every comparator; the winning rule is reported in `optimal_rule`.
- CheckV AF / AAI come from UHVDB metadata (`aai_af`, `aai_id`); AAI × AF = `(aai_id/100) × aai_af`.
- Outputs → `figure_4/plots/figure_4d_revision/` (histograms + violins for |gap| ≥ 10 pp separators; also bulk-vs-HEV and no-MMC-vs-MMC selected violins)
